# Demo 1 — Baseline SAE Polysemanticity (Python-only)

Investor takeaway: A single sparse autoencoder (SAE) trained on small model activations (tiny GPT-2) produces polysemantic features — single features mix multiple concepts. This establishes the baseline “disease” that later demos (ensemble intersections, spike/hypergraph, STII/ACDC) aim to cure.

This notebook reproduces the headless pipeline, saves artifacts, and aligns with the metrics in [METRICS.md](METRICS.md) and framing in [DEMO_CORRIDOR.md](DEMO_CORRIDOR.md).

In [ ]:
import os, sys, random, json
import numpy as np
import torch
import matplotlib.pyplot as plt
import yaml
from transformers import __version__ as transformers_version

# Ensure local modules are importable
sys.path.append(os.path.abspath("."))

from python.utils.config import load_yaml  # noqa: E402
from python.datasets.bank_sentences import generate_bank_dataset  # noqa: E402
from python.activations.extract import get_model_and_tokenizer, capture_layer_activations  # noqa: E402
from python.models.sae import train_sae, encode_topk  # noqa: E402
from python.metrics.polysemanticity import concept_probs, poly_count, entropy, summarize_polysemanticity  # noqa: E402
from python.plots.hist import plot_histogram  # noqa: E402
from python.utils.artifacts import create_run_dir, dump_json, dump_yaml  # noqa: E402

print("env:")
print("  torch:", torch.__version__)
print("  transformers:", transformers_version)
print("  numpy:", np.__version__)
print("  matplotlib:", plt.matplotlib.__version__)
print("  pyyaml:", yaml.__version__)

def _seed_all(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

CFG_PATH = "configs/demo1_baseline.yaml"

In [ ]:
# 1) Load config
cfg = load_yaml(CFG_PATH)
model_name = cfg["model_name"]
layer_index = int(cfg["layer_index"])
ds_cfg = cfg["dataset"]
sae_cfg = cfg["sae"]
metrics_cfg = cfg["metrics"]
out_cfg = cfg["outputs"]
print("config loaded; model=", model_name, "layer_index=", layer_index)

# Determinism
_seed_all(int(ds_cfg.get("seed", 1337)))

In [ ]:
# 2) Generate dataset and preview
texts, labels = generate_bank_dataset(n_per_class=int(ds_cfg["n_per_class"]), seed=int(ds_cfg["seed"]))
labels_np = np.asarray(labels, dtype=np.int32)
num_concepts = len(set(labels))
concept_names = ds_cfg.get("concepts", [f"concept_{i}" for i in range(num_concepts)])

for i in range(5):
    print(f"[{labels[i]}]", texts[i])

In [ ]:
# 3) Extract activations from tiny GPT-2 (MLP input at chosen layer)
model, tokenizer = get_model_and_tokenizer(model_name)
acts = capture_layer_activations(model, tokenizer, texts, layer_index=layer_index)
print("activations shape:", acts.shape)
input_dim = int(acts.shape[1])

In [ ]:
# 4) Train SAE; plot loss curves
_seed_all(int(sae_cfg.get("seed", 42)))
hidden_dim = int(sae_cfg["hidden_dim"])
top_k = int(sae_cfg["top_k"])
epochs = int(sae_cfg["epochs"])
lr = float(sae_cfg["lr"])
l1_lambda = float(sae_cfg["l1_lambda"])
active_threshold = float(sae_cfg["active_threshold"])

sae_model, train_stats = train_sae(
    acts=acts,
    hidden_dim=hidden_dim,
    top_k=top_k,
    l1_lambda=l1_lambda,
    seed=int(sae_cfg.get("seed", 42)),
    epochs=epochs,
    lr=lr,
    device="cpu",
)

plt.figure(figsize=(6,3))
plt.plot(train_stats["total_curve"], label="total")
plt.plot(train_stats["recon_curve"], label="recon")
plt.plot(train_stats["l1_curve"], label="l1")
plt.legend(); plt.title("SAE training loss"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.tight_layout(); plt.show()

In [ ]:
# 5) Encode and compute metrics
features = encode_topk(sae_model, acts, top_k=top_k, device="cpu")  # [N, H]
prob = concept_probs(features, labels_np, num_concepts=num_concepts, active_threshold=active_threshold)  # [H, m]
eps = float(metrics_cfg["eps"])  # poly threshold
poly = poly_count(prob, eps=eps)
ent = entropy(prob)
summary = summarize_polysemanticity(prob, eps=eps)
print(json.dumps(summary, indent=2))

In [ ]:
# 6) Preview histogram via helper (saved to a temp preview path)
os.makedirs(out_cfg["base_dir"], exist_ok=True)
preview_hist = os.path.join(out_cfg["base_dir"], "__preview_poly_hist.png")
title = f"SAE polysemanticity (eps={eps:.2g}) — median={summary['median_poly']:.2f}, monosemantic_rate={summary['monosemantic_rate']:.1%}"
plot_histogram(poly, bins=int(metrics_cfg["hist_bins"]), title=title, path=preview_hist)
print("preview histogram saved:", preview_hist)

### Acceptance criteria mapping
- Metrics follow [METRICS.md](METRICS.md): poly(f) counts concepts with P(C_k | f_active) > ε; entropy uses base-2. When a feature never fires, we fall back to uniform.
- Investor framing from [DEMO_CORRIDOR.md](DEMO_CORRIDOR.md): expect low monosemantic rate and high median polysemanticity for the single-SAE baseline.
- Reproducibility: seeds are taken from config for dataset and SAE training; artifacts include metrics JSON, config copy, arrays, and histogram image.

In [ ]:
# 7) Inspect example polysemantic features (top-3 by entropy)
top3 = np.argsort(ent)[-3:][::-1]
for j in top3:
    row = prob[j]
    pairs = list(zip(concept_names, row.tolist()))
    pairs.sort(key=lambda x: x[1], reverse=True)
    print(f"feature {j} entropy={ent[j]:.3f} poly={poly[j]} probs={pairs}")

In [ ]:
# 8) Save artifacts to a fresh run directory
run_dir = create_run_dir(base_dir=out_cfg["base_dir"], run_tag=out_cfg.get("run_tag"))
np.save(os.path.join(run_dir, "probs.npy"), prob)
np.save(os.path.join(run_dir, "poly_counts.npy"), poly)
np.save(os.path.join(run_dir, "entropy.npy"), ent)
dump_yaml(cfg, os.path.join(run_dir, "config.yaml"))

metrics_out = {
    **summary,
    "eps": float(eps),
    "active_threshold": float(active_threshold),
    "num_features": int(prob.shape[0]),
    "num_concepts": int(prob.shape[1]),
    "concepts": list(concept_names),
}
dump_json(metrics_out, os.path.join(run_dir, "metrics.json"))

hist_path = os.path.join(run_dir, "poly_hist.png")
title = f"SAE polysemanticity (eps={eps:.2g}) — median={summary['median_poly']:.2f}, monosemantic_rate={summary['monosemantic_rate']:.1%}"
plot_histogram(poly, bins=int(metrics_cfg["hist_bins"]), title=title, path=hist_path)
print("artifacts saved to:", run_dir)
print(f"Baseline SAE polysemanticity: median={summary['median_poly']:.2f}, monosemantic_rate={summary['monosemantic_rate']:.3f}")